In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [14]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_panama.json")

### Rules

In [19]:
Rule1 = Rule('''
MATCH (i:Intermediary)-[:intermediary_of]->(e:Entity)
GENERATE
(c = (i, "country"):T_country {
    ccode = i.country_codes,
    name = i.name,
    internal_id = toInteger(i.internal_id)
})<-[():LOCATED_IN]-(x = (i, "director"):T_director {
    name = i.name
})-[():DIRECTOR_OF]->(y = (e):)
''', env=env, type_strict=False)


Rule2 = Rule('''
MATCH (o1:Officer)-[:director_of]->(o2:Officer)-[:director_of*]->()
WITH o1, o2 LIMIT 500
GENERATE
(c = (o1.sourceID):T_country {
    valid = o1.valid_until,
    name = o1.name
})<-[():LOCATED_IN]-(x = (o1):T_director {
    name = o1.name
})
''', env=env, type_strict=False)


Rule3 = Rule('''
MATCH (o1:Officer)-[:officer_of*]->(e:Entity)-[:registered_address]->(a:Address)
WITH o1, e, a LIMIT 2000
GENERATE
(c = (a):T_country {
    ccode = a.country_codes,
    cname = a.countries
})<-[():INTERVENED_IN]-(x = (o1):T_director)
''', env=env, type_strict=False)

Rule4 = Rule('''
MATCH (o1:Intermediary)-[:intermediary_of]->(e:Entity)-[:registered_address]->(a:Address)
WITH o1, e, a LIMIT 2000
GENERATE
(c = (a):T_country {
    ccode = a.country_codes,
    cname = a.countries
})<-[():INTERVENED_IN]-(x = (o1):T_intermediary)
''', env=env, type_strict=False)

Rule5 = Rule('''
MATCH (o1:Entity)-[:officer_of*]->(a:Address)
GENERATE
(c = (a.countryname):T_country {
    ccode = a.country_codes,
    cname = a.countries
})<-[():INTERVENED_IN]-(x = (o1):T_entity)
''', env=env, type_strict=False)

In [ ]:
Rule1._compile(graph.database)
print(Rule1._compiled)

In [16]:
my_transform = Transformation([Rule1])
my_transform.apply_on(graph)

Index: Added 1 index, completed after 11 ms.


Rule: Added 703976 labels, created 650760 nodes, set 4813924 properties, created 625105 relationships, completed after 24806 ms.


24806

### Abort Transformation

In [18]:
my_transform.abort()

TransformationDeactivationError: This transformation is not currently active.